# Model quality estimation

## Task 1. Answer the questions from the introduction
1. What is leave-one-out? Provide limitations and strengths.
  
    Answer: Leave-One-Out cross-validator provides train/test indices to split data in train/test sets. <b>Each sample is used once as a test set</b> (singleton) while the remaining samples form the training set.

    <b>Strengths</b>:
    * Maximum use of data.
    * Preservation of data distribution.
    * Suitable for very small datasets.

    <b>Limitations</b>:
    * High computational complexity.
    * High variance in model performance evaluation.
    * Sensitivity to outliers.
    * Risk of overfitting.
    * Risk of data leakage.
2. How do Grid Search, Randomized Grid Search, and Bayesian optimization work?

    <b>Grid Search</b>: In grid search, the algorithm tries all possible combinations of parameters; after evaluating all combinations, the algorithm selects the combination of parameters that yields the best result according to the evaluation function.

    <b>Randomized Grid Search</b>: In a randomized grid search, the method generates a fixed number of random combinations of hyperparameters by selecting values from specified distributions. For each randomly selected combination of hyperparameters, the model is trained, and its performance is evaluated using a selected metric via cross-validation.

    <b>Bayesian optimization</b>: In Bayesian optimization, initial points are first determined; at each iteration, the probabilistic model is updated based on new data (objective function values at the selected points). An acquisition function is used to determine the point at which the objective function value should be computed next. This point maximizes the objective.  The process is repeated until a specified stopping criterion is met.

3. Explain classification of feature selection methods. Explain how Pearson and Chi2 work. Explain how Lasso works. Explain what permutation significance is. Become familiar with SHAP.

    <b>Pearson</b>: Pearson's correlation coefficient is designed to detect only linear relationships between variables. Pearson's correlation requires that each of the variables being compared be normally distributed. The Pearson correlation coefficient ranges from −1 to +1:
    * +1 — perfect positive correlation (when the value of one variable increases, the value of the other also increases);
    * 0 — no linear correlation;
    * -1 — perfect negative correlation (when the value of one variable increases, the value of the other decreases).

    <b>Chi2</b>: The chi-square test measures the difference between the observed and expected frequencies of categories under the null hypothesis that the variables are independent. The null hypothesis assumes that two categorical variables are independent. The alternative hypothesis is that there is a relationship between the variable and the response variable.

    <b>Lasso</b>: Lasso modifies the objective function (for example, in linear regression) by adding a term that represents the L1 norm of the coefficient vector—the sum of the absolute values of the coefficients. This results in some coefficients being set to zero, which allows the method to perform feature selection.

    <b>Permutation significance</b>: The basic idea behind the method is to generate a "zero" distribution by randomly shuffling the data. This assumes that there are no significant correlations or differences between the variables or groups.
    
    How it works:
      * The original data is divided into two samples.
      * A test statistic is selected that captures the essence of the hypothesis being tested.
      * The values or data groups are randomly shuffled while preserving the structure of the variables.
      * Steps 2 and 3 are repeated many times (often 1,000–10,000 times) to construct a null distribution of the test statistic.
      * Comparing the original test statistic with the null distribution determines the proportion of permutations in which the test statistic was as extreme or more extreme than the observed value.


## Task 2. Introduction — do all the preprocessing from the previous lesson
* Read all the data.


In [1]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.8 MB/s eta 0:00:00


In [51]:
import numpy as np
import pandas as pd

import time
from collections import Counter
import itertools
from tqdm import tqdm

from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, make_scorer

import shap

import optuna
from optuna.samplers import TPESampler

In [3]:
data = pd.read_json('data/train.json')

* Create features: 'Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'LaundryInBuilding', 'NewConstruction', 'Terrace'.

In [4]:
low, high = np.percentile(data['price'], [1, 99])

data = data[(data['price'] >= low) & (data['price'] <= high)]

In [5]:
data.features = data.features.apply(lambda x: ','.join(x).replace(' ', ''))

features_list = []

for _, row in data.iterrows():
  curr = row.features.split(',')
  if curr != ['']:
    features_list.extend(curr)

counter = Counter(features_list)

features = counter.most_common(20)
features_df = pd.DataFrame()

for f in features:
  features_df[f[0]] = data.features.apply(lambda x: int(f[0] in x.split(',')))

features_df['bathrooms'] = data.bathrooms
features_df['bedrooms'] = data.bedrooms

features_df.head()

,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,...,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace,bathrooms,bedrooms
4,0,1,1,1,0,1,0,1,0,1,...,0,1,0,0,0,0,0,0,1.0,1
6,1,1,0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,1.0,2
9,1,1,0,0,1,1,0,1,0,0,...,0,0,0,0,0,0,0,0,1.0,2
10,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1.5,3
15,1,0,0,0,1,0,0,1,1,0,...,0,0,0,0,0,0,0,0,1.0,0


## Task 3. Implement the next methods:
* Split data into 2 parts randomly with parameter test_size (ratio from 0 to 1), return training and test samples.

In [6]:
def my_train_test_split(X, y, test_size=0.25, random_state=21):
  rows = len(X)
  test_count = int(np.round(rows * test_size))

  X_train = X.sample(frac=1 - test_size, random_state=random_state)
  y_train = y.iloc[X_train.index]

  X_test = X.drop(X_train.index)
  y_test = y.drop(X_train.index)

  return X_train, y_train, X_test, y_test

* Randomly split data into 3 parts with parameters validation_size and test_size, return train, validation and test samples.

In [7]:
def my_train_validation_test_split(X, y, validation_size=0.25, test_size=0.25, random_state=21):
  rows = len(X)
  np.random.seed(random_state)

  indices = np.random.permutation(rows)

  test_idx = int(len(X) * (1 - test_size))

  val_idx = int(len(X) * (1 - (validation_size + test_size)))

  train_idx = indices[:val_idx]
  val_idx = indices[val_idx:test_idx]
  test_idx = indices[test_idx:]

  return X.iloc[train_idx], y.iloc[train_idx], X.iloc[val_idx], y.iloc[val_idx], X.iloc[test_idx], y.iloc[test_idx]

* Split data into 2 parts with parameter date_split, return train and test samples split by date_split param.

In [8]:
def my_train_test_date_split(X, y, date_column, date_split, random_state=21):
  X[date_column] = pd.to_datetime(X[date_column])
  date_split = pd.to_datetime(date_split)

  train_idx = X[date_column] < date_split
  test_idx = X[date_column] >= date_split

  X_train = X[train_idx]
  y_train = y[train_idx]
  X_test = X[test_idx]
  y_test = y[test_idx]

  return X_train, y_train, X_test, y_test

* Split data into 3 parts with parameters validation_date and test_date, return train, validation and test samples split by input params.

In [9]:
def my_train_validation_test_date_split(X, y, date_column, validation_date, test_date, random_state=21):
  X[date_column] = pd.to_datetime(X[date_column])
  date_split = pd.to_datetime(date_split)

  train_idx = X[date_column] < validation_date
  val_idx = (X[date_column] >= validation_date) & (X[date_column] < test_date)
  test_idx = X[date_column] >= test_date

  X_train = X[train_idx]
  y_train = y[train_idx]
  X_val = X[val_idx]
  y_val = y[val_idx]
  X_test = X[test_idx]
  y_test = y[test_idx]

  return X_train, y_train, X_val, y_val, X_test, y_test

## Task 4. Implement the next cross-validation methods:
* K-Fold, where k is the input parameter, returns a list of train and test indices.

In [10]:
def my_KFold_split(X, k=5, shuffle=False, random_state=None):
  rows = len(X)
  idx = np.arange(rows)
  if shuffle:
    np.random.seed(random_state)
    np.random.shuffle(idx)

  test_splits = np.array_split(idx, k)

  splits = []

  for test_idx in test_splits:
    train_idx = np.setdiff1d(idx, test_idx)
    splits.append((train_idx, test_idx))

  return splits

* Grouped K-Fold, where k and group_field are input parameters, returns list of train and test indices.

In [11]:
def my_Grouped_KFold_split(X, group_field, k=5, shuffle=False, random_state=None):
  rows = len(X)
  idx = np.arange(rows)
  groups = X[group_field].unique()

  if k > len(groups):
    raise ValueError('k must be less than or equal to groupes count')

  if shuffle:
    if random_state is not None:
      np.random.seed(random_state)
    np.random.shuffle(groups)

  test_groups = np.array_split(groups, k)

  splits = []

  for test_group in test_groups:
    mask = X[group_field].isin(test_group)

    test_positions = np.where(mask)[0]
    train_positions = np.where(~mask)[0]

    splits.append((train_positions, test_positions))

  return splits

* Stratified K-fold, where k and stratify_field are input parameters, returns list of train and test indices.

In [12]:
def my_Stratified_KFold_split(X, stratify_field, k=5, shuffle=False, random_state=None):
  if shuffle and random_state:
        np.random.seed(random_state)

  rows = len(X)
  y = X[stratify_field]
  unique_classes = y.unique()

  index_to_position = {idx: pos for pos, idx in enumerate(X.index)}

  class_positions = {}
  for cls in unique_classes:
    original_idx = y[y == cls].index.tolist()
    positions = [index_to_position[idx] for idx in original_idx]

    if shuffle:
      np.random.shuffle(positions)

    class_positions[cls] = positions

  class_folds = {}
  for cls, positions in class_positions.items():
    class_folds[cls] = np.array_split(positions, k)

  splits = []
  all_pos = np.arange(rows)

  for fold in range(k):
    test_positions = []
    for cls in unique_classes:
      test_positions.extend(class_folds[cls][fold])

    test_positions = np.array(test_positions)

    train_positions = np.setdiff1d(all_pos, test_positions)

    splits.append((train_positions, test_positions))

  return splits

* Time series split, where k and date_field are input parameters, returns list of train and test indices.

In [13]:
def my_TimeSeriesSplit(X, date_field, k=5):
  X_sorted = X.sort_values(date_field).reset_index(drop=True)
  rows = len(X_sorted)

  test_size = rows // (k + 1)

  splits = []

  for i in range(1, k + 1):
    test_start = i  * test_size
    test_end = test_start + test_size

    if i == k:
      test_end = rows

    test_positions = np.arange(test_start, test_end)

    train_positions = np.arange(0, test_start)

    if len(train_positions) > 0:
      splits.append((train_positions, test_positions))

  return splits

## Task 5. Cross-validation comparison
* Apply all the validation methods implemented above to our dataset. To apply Stratified algorithm you should preprocess target.

In [14]:
final_df = pd.concat([features_df, data.price], axis=1)

### K-Fold

In [15]:
for i, (train_idx, test_idx) in enumerate(my_KFold_split(final_df, k=5, shuffle=True, random_state=21)):
  print(f'Fold {i+1}')
  print(f'train:\n\t{train_idx}')
  print(f'test:\n\t{test_idx}')

Fold 1
train:
	[    0     1     3 ... 48372 48374 48376]
test:
	[ 1270 37030 29625 ... 47042 46275 39531]
Fold 2
train:
	[    1     2     3 ... 48376 48377 48378]
test:
	[44789  2875 13110 ... 18512   862 21075]
Fold 3
train:
	[    0     1     2 ... 48375 48377 48378]
test:
	[27870 35325 26603 ...  1800 27879 40136]
Fold 4
train:
	[    0     1     2 ... 48376 48377 48378]
test:
	[43094 35046 15003 ...  8653 41042  6238]
Fold 5
train:
	[    0     2     3 ... 48376 48377 48378]
test:
	[34551  6912 30457 ...  5944  5327 15305]


### Grouped K-Fold

In [16]:
for i, (train_idx, test_idx) in enumerate(my_Grouped_KFold_split(final_df, 'bedrooms', k=5, shuffle=True, random_state=21)):
  print(f'Fold {i+1}')
  print(f'train:\n\t{final_df.bedrooms.iloc[train_idx].unique()}')
  print(f'test:\n\t{final_df.bedrooms.iloc[test_idx].unique()}')

Fold 1
train:
	[1 0 4 5 6 8 7]
test:
	[2 3]
Fold 2
train:
	[1 2 3 4 5 6 7]
test:
	[0 8]
Fold 3
train:
	[1 2 3 0 4 8 7]
test:
	[5 6]
Fold 4
train:
	[2 3 0 5 6 8 7]
test:
	[1 4]
Fold 5
train:
	[1 2 3 0 4 5 6 8]
test:
	[7]


## Stratified K-Fold

In [17]:
bins = [-1, 2500, 4065, 13000]
labels = ['low', 'medium', 'high']

bin_target = pd.cut(final_df.price, bins=bins, labels=labels)

final_df['bin_price'] = bin_target

In [18]:
for i, (train_idx, test_idx) in enumerate(my_Stratified_KFold_split(final_df, 'bin_price', k=3, shuffle=True, random_state=21)):
  train_counts = len(final_df.bin_price.iloc[train_idx])
  test_counts = len(final_df.bin_price.iloc[test_idx])
  print(f'Fold {i+1}')
  print(f'train:\n\t{np.round(final_df.bin_price.iloc[train_idx].value_counts() / train_counts * 100, 2)}')
  print(f'test:\n\t{np.round(final_df.bin_price.iloc[test_idx].value_counts() / test_counts * 100, 2)}\n\n')

Fold 1
train:
	bin_price
medium    48.25
low       26.75
high      25.00
Name: count, dtype: float64
test:
	bin_price
medium    48.25
low       26.75
high      25.00
Name: count, dtype: float64


Fold 2
train:
	bin_price
medium    48.25
low       26.75
high      25.00
Name: count, dtype: float64
test:
	bin_price
medium    48.25
low       26.75
high      25.00
Name: count, dtype: float64


Fold 3
train:
	bin_price
medium    48.25
low       26.75
high      25.00
Name: count, dtype: float64
test:
	bin_price
medium    48.25
low       26.75
high      25.00
Name: count, dtype: float64




### TimeSeriesSplit

In [19]:
final_df['date'] = data.created

In [20]:
sorted_df = final_df.sort_values('date').reset_index(drop=True)

for i, (train_idx, test_idx) in enumerate(my_TimeSeriesSplit(sorted_df, 'date', k=5)):
  print(f'Fold {i+1}')
  print(f'train:\n\tmin: {sorted_df.date.iloc[train_idx].min()} max: {sorted_df.date.iloc[train_idx].max()}')
  print(f'test:\n\tmin: {sorted_df.date.iloc[test_idx].min()} max: {sorted_df.date.iloc[test_idx].max()}')

Fold 1
train:
	min: 2016-04-01 22:12:41 max: 2016-04-16 01:29:59
test:
	min: 2016-04-16 01:30:37 max: 2016-05-02 01:14:08
Fold 2
train:
	min: 2016-04-01 22:12:41 max: 2016-05-02 01:14:08
test:
	min: 2016-05-02 01:14:30 max: 2016-05-16 02:56:35
Fold 3
train:
	min: 2016-04-01 22:12:41 max: 2016-05-16 02:56:35
test:
	min: 2016-05-16 02:56:37 max: 2016-06-01 05:58:11
Fold 4
train:
	min: 2016-04-01 22:12:41 max: 2016-06-01 05:58:11
test:
	min: 2016-06-01 05:58:17 max: 2016-06-15 02:25:53
Fold 5
train:
	min: 2016-04-01 22:12:41 max: 2016-06-15 02:25:53
test:
	min: 2016-06-15 02:26:31 max: 2016-06-29 21:41:47


* Apply the appropriate methods from sklearn.

### K-Fold

In [21]:
kf = KFold(n_splits=5, shuffle=True, random_state=21)
for i, (train_index, test_index) in enumerate(kf.split(final_df)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")

Fold 0:
  Train: index=[    0     1     3 ... 48372 48374 48376]
  Test:  index=[    2     7    21 ... 48375 48377 48378]
Fold 1:
  Train: index=[    1     2     3 ... 48376 48377 48378]
  Test:  index=[    0    11    13 ... 48366 48370 48372]
Fold 2:
  Train: index=[    0     1     2 ... 48375 48377 48378]
  Test:  index=[    4    18    31 ... 48369 48374 48376]
Fold 3:
  Train: index=[    0     1     2 ... 48376 48377 48378]
  Test:  index=[    3     6    14 ... 48358 48362 48367]
Fold 4:
  Train: index=[    0     2     3 ... 48376 48377 48378]
  Test:  index=[    1     5     8 ... 48345 48349 48352]


### Group K-Fold

In [22]:
group_kfold = GroupKFold(n_splits=5, shuffle=True, random_state=21)
groups = final_df['bedrooms']

for i, (train_index, test_index) in enumerate(group_kfold.split(final_df, groups=groups)):
    print(f"Fold {i}:")
    print(f"  Train: group={groups.iloc[train_index].unique()}")
    print(f"  Test:  group={groups.iloc[test_index].unique()}")

Fold 0:
  Train: group=[3 0 4 5 6 8 7]
  Test:  group=[1 2]
Fold 1:
  Train: group=[1 2 0 4 5 6 8]
  Test:  group=[3 7]
Fold 2:
  Train: group=[1 2 3 0 4 8 7]
  Test:  group=[5 6]
Fold 3:
  Train: group=[1 2 3 5 6 8 7]
  Test:  group=[0 4]
Fold 4:
  Train: group=[1 2 3 0 4 5 6 7]
  Test:  group=[8]


### Stratified K-Fold

In [23]:
skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=21)
y_bin = final_df['bin_price']

for i, (train_index, test_index) in enumerate(skf.split(final_df, y_bin)):
  train_counts = len(y_bin.iloc[train_idx])
  test_counts = len(y_bin.iloc[test_idx])
  print(f'Fold {i+1}')
  print(f'train:\n\t{np.round(y_bin.iloc[train_idx].value_counts() / train_counts * 100, 2)}')
  print(f'test:\n\t{np.round(y_bin.iloc[test_idx].value_counts() / test_counts * 100, 2)}\n\n')

Fold 1
train:
	bin_price
medium    48.22
low       26.63
high      25.15
Name: count, dtype: float64
test:
	bin_price
medium    48.39
low       27.39
high      24.22
Name: count, dtype: float64


Fold 2
train:
	bin_price
medium    48.22
low       26.63
high      25.15
Name: count, dtype: float64
test:
	bin_price
medium    48.39
low       27.39
high      24.22
Name: count, dtype: float64




### TimeSeriesSplit

In [24]:
tscv = TimeSeriesSplit()

for i, (train_index, test_index) in enumerate(tscv.split(sorted_df)):
  print(f'Fold {i+1}')
  print(f"  Train: index={train_index}")
  print(f"  Test:  index={test_index}")

Fold 1
  Train: index=[   0    1    2 ... 8061 8062 8063]
  Test:  index=[ 8064  8065  8066 ... 16124 16125 16126]
Fold 2
  Train: index=[    0     1     2 ... 16124 16125 16126]
  Test:  index=[16127 16128 16129 ... 24187 24188 24189]
Fold 3
  Train: index=[    0     1     2 ... 24187 24188 24189]
  Test:  index=[24190 24191 24192 ... 32250 32251 32252]
Fold 4
  Train: index=[    0     1     2 ... 32250 32251 32252]
  Test:  index=[32253 32254 32255 ... 40313 40314 40315]
Fold 5
  Train: index=[    0     1     2 ... 40313 40314 40315]
  Test:  index=[40316 40317 40318 ... 48376 48377 48378]


In [25]:
for i, (train_idx, test_idx) in enumerate(my_TimeSeriesSplit(sorted_df, 'date', k=5)):
  print(f'Fold {i+1}')
  print(f"  Train: index={train_idx}")
  print(f"  Test:  index={test_idx}")

Fold 1
  Train: index=[   0    1    2 ... 8060 8061 8062]
  Test:  index=[ 8063  8064  8065 ... 16123 16124 16125]
Fold 2
  Train: index=[    0     1     2 ... 16123 16124 16125]
  Test:  index=[16126 16127 16128 ... 24186 24187 24188]
Fold 3
  Train: index=[    0     1     2 ... 24186 24187 24188]
  Test:  index=[24189 24190 24191 ... 32249 32250 32251]
Fold 4
  Train: index=[    0     1     2 ... 32249 32250 32251]
  Test:  index=[32252 32253 32254 ... 40312 40313 40314]
Fold 5
  Train: index=[    0     1     2 ... 40312 40313 40314]
  Test:  index=[40315 40316 40317 ... 48376 48377 48378]


* Compare the resulting feature distributions for the training part of the dataset between sklearn and your implementation.

### K-Fold

In [26]:
data = []

for i, (train_idx, test_idx) in enumerate(my_KFold_split(final_df, k=5, shuffle=True, random_state=21)):
  for col in final_df.columns:
    if col not in ['bin_price', 'date']:
      my_data = final_df.iloc[train_idx][col]
      stats_my = {
          'fold': i,
          'feature': col + '_my',
          'mean': my_data.mean(),
          'std': my_data.std(),
          'min': my_data.min(),
          'max': my_data.max(),
          'q25': my_data.quantile(0.25),
          'q50': my_data.quantile(0.50),
          'q75': my_data.quantile(0.75)
        }

      data.append(stats_my)

kf = KFold(n_splits=5, shuffle=True, random_state=21)
for i, (train_index, test_index) in enumerate(kf.split(final_df)):

  for col in final_df.columns:
    if col not in ['bin_price', 'date']:
      sk_data = final_df.iloc[train_idx][col]
      stats_sk = {
          'fold': i,
          'feature': col + '_sklearn',
          'mean': sk_data.mean(),
          'std': sk_data.std(),
          'min': sk_data.min(),
          'max': sk_data.max(),
          'q25': sk_data.quantile(0.25),
          'q50': sk_data.quantile(0.50),
          'q75': sk_data.quantile(0.75)
        }
      data.append(stats_sk)

kfold_distr = pd.DataFrame(data).sort_values(by=['fold', 'feature'])

kfold_distr[kfold_distr['fold'] == 0]

,fold,feature,mean,std,min,max,q25,q50,q75
15,0,Balcony_my,0.060409,0.238246,0.0,1.0,0.0,0.0,0.0
130,0,Balcony_sklearn,0.060459,0.238338,0.0,1.0,0.0,0.0,0.0
2,0,CatsAllowed_my,0.477612,0.499505,0.0,1.0,0.0,0.0,1.0
117,0,CatsAllowed_sklearn,0.477909,0.499518,0.0,1.0,0.0,0.0,1.0
13,0,DiningRoom_my,0.101491,0.301981,0.0,1.0,0.0,0.0,0.0
128,0,DiningRoom_sklearn,0.102393,0.303168,0.0,1.0,0.0,0.0,0.0
5,0,Dishwasher_my,0.414051,0.492564,0.0,1.0,0.0,0.0,1.0
120,0,Dishwasher_sklearn,0.415461,0.492808,0.0,1.0,0.0,0.0,1.0
3,0,DogsAllowed_my,0.447020,0.497192,0.0,1.0,0.0,0.0,1.0
118,0,DogsAllowed_sklearn,0.447602,0.497253,0.0,1.0,0.0,0.0,1.0


* Compare all validation schemes. Choose the best one. Explain your choice.

    K-fold cross-validation is suitable for this task; stratified K-fold cross-validation is useful when the target variable has a long-tail distribution, in which case the variable must be binarized

## Task 6. Feature Selection
* Fit a Lasso regression model with normalized features. Use your method for splitting samples into 3 parts by field created with 60/20/20 ratio — train/validation/test.

In [27]:
def get_metrics(model, X_train, X_val, X_test, y_train, y_val, y_test, name):
  model.fit(X_train, y_train)

  y_train_pred = model.predict(X_train)
  y_val_pred = model.predict(X_val)
  y_test_pred = model.predict(X_test)

  mae_result ={
      'model': name,
      'train': mean_absolute_error(y_train, y_train_pred),
      'valid': mean_absolute_error(y_val, y_val_pred),
      'test': mean_absolute_error(y_test, y_test_pred),
  }

  rmse_result = {
      'model': name,
      'train': root_mean_squared_error(y_train, y_train_pred),
      'valid': root_mean_squared_error(y_val, y_val_pred),
      'test': root_mean_squared_error(y_test, y_test_pred),
  }

  r2_result = {
      'model': name,
      'train': r2_score(y_train, y_train_pred),
      'valid': r2_score(y_val, y_val_pred),
      'test': r2_score(y_test, y_test_pred),
  }

  return mae_result, rmse_result, r2_result

In [28]:
final_df = final_df.drop(['bin_price'], axis=1)

In [29]:
final_df.rename(columns={'date': 'created'}, inplace=True)

In [30]:
sorted_df = final_df.sort_values('created').reset_index(drop=True).copy()
X = sorted_df.drop(['price', 'created'], axis=1).copy()
y = sorted_df['price'].copy()

X_train, y_train, X_val, y_val, X_test, y_test = my_train_validation_test_split(X, y, validation_size=0.2, test_size=0.2, random_state=21)

In [31]:
features_to_scale = ['bedrooms', 'bathrooms']

numerical_transformer = ColumnTransformer(
    transformers=[
        ('scaler', MinMaxScaler(), features_to_scale)
    ],
    remainder='passthrough'
)

numerical_transformer.fit(X_train)

X_train_scaled = numerical_transformer.transform(X_train)
X_val_scaled = numerical_transformer.transform(X_val)
X_test_scaled = numerical_transformer.transform(X_test)

In [32]:
lasso_model = Lasso()

mae, rmse, r2 = get_metrics(lasso_model, X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test, 'Lasso MinMaxScaler')

mae_result = [mae]

rmse_result = [rmse]

r2_result = [r2]

* Sort features by weight coefficients from model, fit model to top 10 features and compare quality.

In [33]:
feature_names = [x.split('__')[1] for x in numerical_transformer.get_feature_names_out()]

top_10_features = pd.DataFrame(np.abs(lasso_model.coef_), feature_names, columns=['weight']).sort_values('weight', ascending=False).head(10)

top_10_features

,weight
bathrooms,6315.213474
bedrooms,3634.052484
Doorman,612.350342
LaundryinUnit,460.292979
LaundryInBuilding,334.053415
FitnessCenter,242.633769
Elevator,226.198078
LaundryinBuilding,222.859173
HighSpeedInternet,221.172753
HardwoodFloors,171.941605


In [34]:
X_train_top_10_scaled = numerical_transformer.fit_transform(X_train[top_10_features.index])
X_val_top_10_scaled = numerical_transformer.transform(X_val[top_10_features.index])
X_test_top_10_scaled = numerical_transformer.transform(X_test[top_10_features.index])

mae, rmse, r2 = get_metrics(lasso_model, X_train_top_10_scaled, X_val_top_10_scaled, X_test_top_10_scaled, y_train, y_val, y_test, 'Lasso top10 MinMaxScaler')

mae_result.append(mae)

rmse_result.append(rmse)

r2_result.append(r2)

In [35]:
pd.DataFrame(mae_result)

,model,train,valid,test
0,Lasso MinMaxScaler,718.020991,699.627626,708.547348
1,Lasso top10 MinMaxScaler,721.948946,702.591138,712.520169


In [35]:
pd.DataFrame(rmse_result)

,model,train,valid,test
0,Lasso MinMaxScaler,1042.813616,1020.104956,1029.388570
1,Lasso top10 MinMaxScaler,1048.584488,1025.830884,1035.227789


In [36]:
pd.DataFrame(r2_result)

,model,train,valid,test
0,Lasso MinMaxScaler,0.581429,0.565065,0.588243
1,Lasso top10 MinMaxScaler,0.576784,0.560168,0.583558


* Implement method for simple feature selection by nan-ratio in feature and correlation. Apply this method to feature set and take top 10 features, refit model and measure quality.

In [36]:
def select_features(X, y, k=10, nan_threshold=0.5):
  nan_ratio = X.isnull().mean()

  non_nan_features = nan_ratio[nan_ratio <= nan_threshold].index.to_list()

  X_selected = X[non_nan_features]

  selected_features = np.abs(X_selected.corrwith(y)).nlargest(k).index.to_list()

  return selected_features

In [37]:
selected_features = select_features(X_train, y_train, k=10)

X_train_top_10_selected = numerical_transformer.fit_transform(X_train[selected_features])
X_val_top_10_selected = numerical_transformer.transform(X_val[selected_features])
X_test_top_10_selected = numerical_transformer.transform(X_test[selected_features])

mae, rmse, r2 = get_metrics(lasso_model, X_train_top_10_selected, X_val_top_10_selected, X_test_top_10_selected, y_train, y_val, y_test, 'Lasso selected MinMaxScaler')

mae_result.append(mae)

rmse_result.append(rmse)

r2_result.append(r2)

In [38]:
display(pd.DataFrame(mae_result))
display(pd.DataFrame(rmse_result))
display(pd.DataFrame(r2_result))

,model,train,valid,test
0,Lasso MinMaxScaler,718.020991,699.627626,708.547348
1,Lasso top10 MinMaxScaler,721.948946,702.591138,712.520169
2,Lasso selected MinMaxScaler,728.618182,710.274463,720.935460


,model,train,valid,test
0,Lasso MinMaxScaler,1042.813616,1020.104956,1029.388570
1,Lasso top10 MinMaxScaler,1048.584488,1025.830884,1035.227789
2,Lasso selected MinMaxScaler,1055.403770,1033.267363,1040.900072


,model,train,valid,test
0,Lasso MinMaxScaler,0.581429,0.565065,0.588243
1,Lasso top10 MinMaxScaler,0.576784,0.560168,0.583558
2,Lasso selected MinMaxScaler,0.571261,0.553768,0.578982


* Implement permutation importance method and take top 10 features, refit model and measure quality.

In [39]:
def select_features_permutation(model, X, y, metric=r2_score, k=10, random_state=21):
  np.random.seed(random_state)
  model.fit(X, y)
  base_metric = metric(y, model.predict(X))
  result = []
  for col in X.columns:
    X_permuted = X.copy()
    X_permuted[col] = np.random.permutation(X[col].values)

    permuted_metric = metric(y, model.predict(X_permuted))

    importance = np.abs(base_metric - permuted_metric)

    result.append({
        'feature': col,
        'importance': importance
    })

  return pd.DataFrame(result).sort_values('importance', ascending=False).head(k)

In [40]:
permutation_features = select_features_permutation(lasso_model, X_train, y_train, metric=mean_absolute_error)['feature'].values

X_train_top_10_permutation = numerical_transformer.fit_transform(X_train[permutation_features])
X_val_top_10_permutation = numerical_transformer.transform(X_val[permutation_features])
X_test_top_10_permutation = numerical_transformer.transform(X_test[permutation_features])

mae, rmse, r2 = get_metrics(lasso_model, X_train_top_10_permutation, X_val_top_10_permutation, X_test_top_10_permutation, y_train, y_val, y_test, 'Lasso permutation MinMaxScaler')

mae_result.append(mae)

rmse_result.append(rmse)

r2_result.append(r2)

In [41]:
display(pd.DataFrame(mae_result))
display(pd.DataFrame(rmse_result))
display(pd.DataFrame(r2_result))

,model,train,valid,test
0,Lasso MinMaxScaler,718.020991,699.627626,708.547348
1,Lasso top10 MinMaxScaler,721.948946,702.591138,712.520169
2,Lasso selected MinMaxScaler,728.618182,710.274463,720.935460
3,Lasso permutation MinMaxScaler,723.149557,705.769389,713.728371


,model,train,valid,test
0,Lasso MinMaxScaler,1042.813616,1020.104956,1029.388570
1,Lasso top10 MinMaxScaler,1048.584488,1025.830884,1035.227789
2,Lasso selected MinMaxScaler,1055.403770,1033.267363,1040.900072
3,Lasso permutation MinMaxScaler,1049.370481,1027.798426,1035.053759


,model,train,valid,test
0,Lasso MinMaxScaler,0.581429,0.565065,0.588243
1,Lasso top10 MinMaxScaler,0.576784,0.560168,0.583558
2,Lasso selected MinMaxScaler,0.571261,0.553768,0.578982
3,Lasso permutation MinMaxScaler,0.576149,0.558479,0.583698


* Import Shap and also refit model on top 10 features.

In [42]:
lasso_model.fit(X_train, y_train)

explainer = shap.Explainer(lasso_model, X_train)

shap_values = explainer(X_val)

shap_importance = np.abs(shap_values.values).mean(axis=0)

importance_df = pd.DataFrame({
    'feature': X_val.columns,
    'shap_importance': shap_importance
})

top_10 = importance_df.sort_values('shap_importance', ascending=False).head(10).feature.values

In [43]:
def select_features_shap(model, X_train, y_train, X_val, k=10):
  model.fit(X_train, y_train)

  explainer = shap.Explainer(model, X_train)

  shap_values = explainer(X_val)

  shap_importance = np.abs(shap_values.values).mean(axis=0)

  importance_df = pd.DataFrame({
      'feature': X_val.columns,
      'shap_importance': shap_importance
  })

  top_10 = importance_df.sort_values('shap_importance', ascending=False).head(10).feature.values
  return top_10

In [44]:
X_train_top_10_shap = numerical_transformer.fit_transform(X_train[top_10])
X_val_top_10_shap = numerical_transformer.transform(X_val[top_10])
X_test_top_10_shap = numerical_transformer.transform(X_test[top_10])

mae, rmse, r2 = get_metrics(lasso_model, X_train_top_10_shap, X_val_top_10_shap, X_test_top_10_shap, y_train, y_val, y_test, 'Lasso Shap MinMaxScaler')

mae_result.append(mae)

rmse_result.append(rmse)

r2_result.append(r2)

In [45]:
display(pd.DataFrame(mae_result))
display(pd.DataFrame(rmse_result))
display(pd.DataFrame(r2_result))

,model,train,valid,test
0,Lasso MinMaxScaler,718.020991,699.627626,708.547348
1,Lasso top10 MinMaxScaler,721.948946,702.591138,712.520169
2,Lasso selected MinMaxScaler,728.618182,710.274463,720.935460
3,Lasso permutation MinMaxScaler,723.149557,705.769389,713.728371
4,Lasso Shap MinMaxScaler,723.641017,706.368279,714.045588


,model,train,valid,test
0,Lasso MinMaxScaler,1042.813616,1020.104956,1029.388570
1,Lasso top10 MinMaxScaler,1048.584488,1025.830884,1035.227789
2,Lasso selected MinMaxScaler,1055.403770,1033.267363,1040.900072
3,Lasso permutation MinMaxScaler,1049.370481,1027.798426,1035.053759
4,Lasso Shap MinMaxScaler,1050.173022,1028.177913,1035.355058


,model,train,valid,test
0,Lasso MinMaxScaler,0.581429,0.565065,0.588243
1,Lasso top10 MinMaxScaler,0.576784,0.560168,0.583558
2,Lasso selected MinMaxScaler,0.571261,0.553768,0.578982
3,Lasso permutation MinMaxScaler,0.576149,0.558479,0.583698
4,Lasso Shap MinMaxScaler,0.575500,0.558153,0.583455


* Compare the quality of these methods for different aspects — speed, metrics and stability.

In [46]:
def select_features_lasso(model, X, k=10):
  importance = np.abs(model.coef_)
  importance_df = pd.Series(importance, index=X.columns)
  return importance_df.nlargest(k).index.tolist()

In [47]:
def compare_speed(X_train, y_train, X_val):

    model = Lasso()
    model.fit(X_train, y_train)

    methods = {
        'Correlation': lambda: select_features(X_train, y_train, k=10),
        'Permutation': lambda: select_features_permutation(model, X_train, y_train, k=10),
        'SHAP': lambda: select_features_shap(model, X_train, y_train, X_val, k=10),
        'Lasso Coef': lambda: select_features_lasso(model, X_train, k=10)
    }

    speed_results = []

    for name, method in methods.items():
        start_time = time.time()
        selected = method()
        elapsed_time = time.time() - start_time
        speed_results.append({
            'Method': name,
            'Time (seconds)': elapsed_time,
            'Selected Features': len(selected)
        })

    return pd.DataFrame(speed_results)

In [48]:
compare_speed(X_train, y_train, X_val)

,Method,Time (seconds),Selected Features
0,Correlation,0.027751,10
1,Permutation,0.260423,10
2,SHAP,0.381900,10
3,Lasso Coef,0.000879,10


## Task 7. Hyperparameter optimization
* Implement grid search and random search methods for alpha and l1_ratio for sklearn's ElasticNet model

In [103]:
def myGridSearch(model, param_grid, X, y, metric=r2_score, random_state=21):
  result = []
  keys = param_grid.keys()
  values = param_grid.values()
  combinations = list(itertools.product(*values))
  scorer = make_scorer(metric)

  model_results = []
  for combination in tqdm(combinations):
    params = dict(zip(keys, combination))
    model_instance = model(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
    scores = cross_val_score(model_instance, X, y, cv=kf, scoring=scorer)

    model_results.append({
        'params': params,
        'mean_score': np.mean(scores),
        'std_score': np.std(scores)
    })

  return model_results

In [104]:
def myRandomGridSearch(model, param_grid, X, y, n_iter=20, metric=r2_score, random_state=21):
  np.random.seed(random_state)
  scorer = make_scorer(metric)
  results = []

  keys = list(param_grid.keys())
  all_values = list(param_grid.values())

  combinations = []
  for _ in range(n_iter):
    combination = [np.random.choice(values) for values in all_values]
    combinations.append(combination)

  for combination in tqdm(combinations, desc="Random Search"):
    params = dict(zip(keys, combination))
    model_instance = model(**params)

    kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
    scores = cross_val_score(model_instance, X, y, cv=kf, scoring=scorer)

    results.append({
        'params': params,
        'mean_score': np.mean(scores),
        'std_score': np.std(scores)
    })

  results.sort(key=lambda x: x['mean_score'], reverse=True)

  return results

* Find the best combination of model hyperparameters.

In [105]:
param_grid = {
    'alpha': [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    'l1_ratio': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
}

result = myGridSearch(ElasticNet, param_grid, X_train_scaled, y_train, metric=mean_absolute_error)

100%|██████████| 90/90 [00:59<00:00,  1.51it/s]


In [106]:
pd.DataFrame(result).sort_values('mean_score')

,params,mean_score,std_score
69,"{'alpha': 2.0, 'l1_ratio': 1.0}",718.629649,4.926650
59,"{'alpha': 1.0, 'l1_ratio': 1.0}",718.676271,4.889844
49,"{'alpha': 0.5, 'l1_ratio': 1.0}",718.813782,4.841457
39,"{'alpha': 0.1, 'l1_ratio': 1.0}",718.936700,4.824225
29,"{'alpha': 0.05, 'l1_ratio': 1.0}",718.959040,4.824918
...,...,...,...
84,"{'alpha': 10.0, 'l1_ratio': 0.5}",1130.366709,5.315704
83,"{'alpha': 10.0, 'l1_ratio': 0.4}",1133.017171,5.370615
82,"{'alpha': 10.0, 'l1_ratio': 0.3}",1135.008782,5.406171
81,"{'alpha': 10.0, 'l1_ratio': 0.2}",1136.562017,5.434270


In [107]:
random_result = myRandomGridSearch(ElasticNet, param_grid, X_train_scaled, y_train)

Random Search: 100%|██████████| 20/20 [00:05<00:00,  3.45it/s]


In [109]:
pd.DataFrame(result).sort_values('mean_score')

,params,mean_score,std_score
69,"{'alpha': 2.0, 'l1_ratio': 1.0}",718.629649,4.926650
59,"{'alpha': 1.0, 'l1_ratio': 1.0}",718.676271,4.889844
49,"{'alpha': 0.5, 'l1_ratio': 1.0}",718.813782,4.841457
39,"{'alpha': 0.1, 'l1_ratio': 1.0}",718.936700,4.824225
29,"{'alpha': 0.05, 'l1_ratio': 1.0}",718.959040,4.824918
...,...,...,...
84,"{'alpha': 10.0, 'l1_ratio': 0.5}",1130.366709,5.315704
83,"{'alpha': 10.0, 'l1_ratio': 0.4}",1133.017171,5.370615
82,"{'alpha': 10.0, 'l1_ratio': 0.3}",1135.008782,5.406171
81,"{'alpha': 10.0, 'l1_ratio': 0.2}",1136.562017,5.434270


* Fit the resulting model.

In [110]:
best_params = result[0]['params']

elastic_model = ElasticNet(**best_params)

mae, rmse, r2 = get_metrics(elastic_model, X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test, 'ElasticNet MinMaxScaler')

mae_result.append(mae)

rmse_result.append(rmse)

r2_result.append(r2)

In [111]:
display(pd.DataFrame(mae_result))
display(pd.DataFrame(rmse_result))
display(pd.DataFrame(r2_result))

,model,train,valid,test
0,Lasso MinMaxScaler,718.020991,699.627626,708.547348
1,Lasso top10 MinMaxScaler,721.948946,702.591138,712.520169
2,Lasso selected MinMaxScaler,728.618182,710.274463,720.935460
3,Lasso permutation MinMaxScaler,723.149557,705.769389,713.728371
4,Lasso Shap MinMaxScaler,723.641017,706.368279,714.045588
5,ElasticNet MinMaxScaler,719.134295,700.584664,709.751940


,model,train,valid,test
0,Lasso MinMaxScaler,1042.813616,1020.104956,1029.388570
1,Lasso top10 MinMaxScaler,1048.584488,1025.830884,1035.227789
2,Lasso selected MinMaxScaler,1055.403770,1033.267363,1040.900072
3,Lasso permutation MinMaxScaler,1049.370481,1027.798426,1035.053759
4,Lasso Shap MinMaxScaler,1050.173022,1028.177913,1035.355058
5,ElasticNet MinMaxScaler,1043.734289,1019.371044,1031.208091


,model,train,valid,test
0,Lasso MinMaxScaler,0.581429,0.565065,0.588243
1,Lasso top10 MinMaxScaler,0.576784,0.560168,0.583558
2,Lasso selected MinMaxScaler,0.571261,0.553768,0.578982
3,Lasso permutation MinMaxScaler,0.576149,0.558479,0.583698
4,Lasso Shap MinMaxScaler,0.575500,0.558153,0.583455
5,ElasticNet MinMaxScaler,0.580690,0.565690,0.586786


* Import optuna and configure the same experiment with ElasticNet.

In [53]:
study = optuna.create_study(study_name="my_first_study", directions=["minimize", 'minimize', 'maximize'])

def objective(trial, X, y):
  alpha = trial.suggest_float('alpha', 0.001, 10.0, log=True)
  l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)


  model = ElasticNet(
    alpha=alpha,
    l1_ratio=l1_ratio,
    max_iter=10000,
    random_state=21
  )


  kf = KFold(n_splits=5, shuffle=True, random_state=21)
  scorer = make_scorer(r2_score)
  scores = cross_val_score(model, X, y, cv=kf, scoring=scorer)


  return scores.mean()

study = optuna.create_study(
    direction='maximize',  # максимизируем R²
    sampler=TPESampler(seed=21)  # TPE - алгоритм байесовской оптимизации
)

study.optimize(
    lambda trial: objective(trial, X_train_scaled, y_train),
    n_trials=100,
    show_progress_bar=True
)

[I 2026-04-09 18:38:20,495] A new study created in memory with name: my_first_study
[I 2026-04-09 18:38:20,504] A new study created in memory with name: no-name-d165ab8c-6351-4831-ab2e-cf4a78918b18


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-04-09 18:38:27,008] Trial 0 finished with value: 0.5788408417876443 and parameters: {'alpha': 0.001566388634308505, 'l1_ratio': 0.28910965978981684}. Best is trial 0 with value: 0.5788408417876443.
[I 2026-04-09 18:38:27,195] Trial 1 finished with value: 0.11711185748534378 and parameters: {'alpha': 0.7653593415503201, 'l1_ratio': 0.021616249915949792}. Best is trial 0 with value: 0.5788408417876443.
[I 2026-04-09 18:38:28,349] Trial 2 finished with value: 0.5563153813921051 and parameters: {'alpha': 0.006663325994391817, 'l1_ratio': 0.0507732566953768}. Best is trial 0 with value: 0.5788408417876443.
[I 2026-04-09 18:38:29,020] Trial 3 finished with value: 0.5610518861029339 and parameters: {'alpha': 0.016184063577845232, 'l1_ratio': 0.6639102946247}. Best is trial 0 with value: 0.5788408417876443.
[I 2026-04-09 18:38:30,425] Trial 4 finished with value: 0.5519553449841693 and parameters: {'alpha': 0.017078808657853408, 'l1_ratio': 0.5835912762185987}. Best is trial 0 with val

In [56]:
print("Number of finished trials: ", len(study.trials))
print("Best trial:")
trial = study.best_trial

trial.params

Number of finished trials:  100
Best trial:


{'alpha': 0.008605990983240072, 'l1_ratio': 0.9958409935421333}

In [57]:
elastic_model = ElasticNet(**trial.params)

mae, rmse, r2 = get_metrics(elastic_model, X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test, 'ElasticNet optuna MinMaxScaler')

mae_result.append(mae)

rmse_result.append(rmse)

r2_result.append(r2)

In [58]:
display(pd.DataFrame(mae_result))
display(pd.DataFrame(rmse_result))
display(pd.DataFrame(r2_result))

,model,train,valid,test
0,Lasso MinMaxScaler,718.020991,699.627626,708.547348
1,Lasso top10 MinMaxScaler,721.948946,702.591138,712.520169
2,Lasso selected MinMaxScaler,728.618182,710.274463,720.935460
3,Lasso permutation MinMaxScaler,723.149557,705.769389,713.728371
4,Lasso Shap MinMaxScaler,723.641017,706.368279,714.045588
5,ElasticNet optuna MinMaxScaler,718.283708,700.124746,708.777220


,model,train,valid,test
0,Lasso MinMaxScaler,1042.813616,1020.104956,1029.388570
1,Lasso top10 MinMaxScaler,1048.584488,1025.830884,1035.227789
2,Lasso selected MinMaxScaler,1055.403770,1033.267363,1040.900072
3,Lasso permutation MinMaxScaler,1049.370481,1027.798426,1035.053759
4,Lasso Shap MinMaxScaler,1050.173022,1028.177913,1035.355058
5,ElasticNet optuna MinMaxScaler,1042.618016,1020.019125,1029.296690


,model,train,valid,test
0,Lasso MinMaxScaler,0.581429,0.565065,0.588243
1,Lasso top10 MinMaxScaler,0.576784,0.560168,0.583558
2,Lasso selected MinMaxScaler,0.571261,0.553768,0.578982
3,Lasso permutation MinMaxScaler,0.576149,0.558479,0.583698
4,Lasso Shap MinMaxScaler,0.575500,0.558153,0.583455
5,ElasticNet optuna MinMaxScaler,0.581586,0.565138,0.588316


* Run optuna on one of the cross-validation schemes.

### Group K-Fold

In [63]:
def objective_groupkfold(trial, X, y, groups, n_splits=5):
  alpha = trial.suggest_float('alpha', 0.001, 10.0, log=True)
  l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)

  model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=21, max_iter=10000)

  gkf = GroupKFold(n_splits=n_splits)

  scores = []
  for train_idx, val_idx in gkf.split(X, y, groups):
    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_train_fold, y_train_fold)
    score = r2_score(y_val_fold, model.predict(X_val_fold))
    scores.append(score)

  return np.mean(scores)

groups = X_train['bedrooms']

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=21)
)

study.optimize(
    lambda trial: objective_groupkfold(trial, X_train_scaled, y_train, groups),
    n_trials=100,
    show_progress_bar=True
)

[I 2026-04-09 19:04:07,138] A new study created in memory with name: no-name-5e0afddf-f9a5-4e1e-a742-c75b6444798a


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-04-09 19:04:08,597] Trial 0 finished with value: 0.2877636592433418 and parameters: {'alpha': 0.001566388634308505, 'l1_ratio': 0.28910965978981684}. Best is trial 0 with value: 0.2877636592433418.
[I 2026-04-09 19:04:08,764] Trial 1 finished with value: -1.0307634945312176 and parameters: {'alpha': 0.7653593415503201, 'l1_ratio': 0.021616249915949792}. Best is trial 0 with value: 0.2877636592433418.
[I 2026-04-09 19:04:09,455] Trial 2 finished with value: 0.21676706273682708 and parameters: {'alpha': 0.006663325994391817, 'l1_ratio': 0.0507732566953768}. Best is trial 0 with value: 0.2877636592433418.
[I 2026-04-09 19:04:12,978] Trial 3 finished with value: 0.2321460616927808 and parameters: {'alpha': 0.016184063577845232, 'l1_ratio': 0.6639102946247}. Best is trial 0 with value: 0.2877636592433418.
[I 2026-04-09 19:04:20,988] Trial 4 finished with value: 0.20250833836077975 and parameters: {'alpha': 0.017078808657853408, 'l1_ratio': 0.5835912762185987}. Best is trial 0 with v

In [65]:
trial = study.best_trial

elastic_model = ElasticNet(**trial.params)

mae, rmse, r2 = get_metrics(elastic_model, X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test, 'ElasticNet optuna GroupKFold')

mae_optuna.append(mae)

rmse_optuna.append(rmse)

r2_optuna.append(r2)

### TimeSeriesSplit

In [68]:
def objective_timeseries(trial, X, y, n_splits=5):
  alpha = trial.suggest_float('alpha', 0.001, 10.0, log=True)
  l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)

  model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=10000)

  tscv = TimeSeriesSplit(n_splits=n_splits)

  scores = []
  for train_idx, val_idx in tscv.split(X):
    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_train_fold, y_train_fold)
    score = r2_score(y_val_fold, model.predict(X_val_fold))
    scores.append(score)

  return np.mean(scores)

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=21)
)

study.optimize(
    lambda trial: objective_timeseries(trial, X_train_scaled, y_train),
    n_trials=100,
    show_progress_bar=True
)

[I 2026-04-09 19:19:42,686] A new study created in memory with name: no-name-f5a97580-def4-4374-830d-660853e5a88e


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-04-09 19:19:42,935] Trial 0 finished with value: 0.582147320554042 and parameters: {'alpha': 0.001566388634308505, 'l1_ratio': 0.28910965978981684}. Best is trial 0 with value: 0.582147320554042.
[I 2026-04-09 19:19:43,008] Trial 1 finished with value: 0.11840575421747808 and parameters: {'alpha': 0.7653593415503201, 'l1_ratio': 0.021616249915949792}. Best is trial 0 with value: 0.582147320554042.
[I 2026-04-09 19:19:43,239] Trial 2 finished with value: 0.5597358193131572 and parameters: {'alpha': 0.006663325994391817, 'l1_ratio': 0.0507732566953768}. Best is trial 0 with value: 0.582147320554042.
[I 2026-04-09 19:19:43,410] Trial 3 finished with value: 0.5644344199093951 and parameters: {'alpha': 0.016184063577845232, 'l1_ratio': 0.6639102946247}. Best is trial 0 with value: 0.582147320554042.
[I 2026-04-09 19:19:43,566] Trial 4 finished with value: 0.555414156051636 and parameters: {'alpha': 0.017078808657853408, 'l1_ratio': 0.5835912762185987}. Best is trial 0 with value: 0.

In [69]:
trial = study.best_trial

elastic_model = ElasticNet(**trial.params)

mae, rmse, r2 = get_metrics(elastic_model, X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test, 'ElasticNet optuna TimeSeriesSplit')

mae_optuna.append(mae)

rmse_optuna.append(rmse)

r2_optuna.append(r2)

### Results

In [70]:
display(pd.DataFrame(mae_optuna))
display(pd.DataFrame(rmse_optuna))
display(pd.DataFrame(r2_optuna))

,model,train,valid,test
0,ElasticNet optuna MinMaxScaler,718.283708,700.124746,708.777220
1,ElasticNet optuna GroupKFold,718.295393,700.123813,708.797102
2,ElasticNet optuna TimeSeriesSplit,718.188507,699.936753,708.698962


,model,train,valid,test
0,ElasticNet optuna MinMaxScaler,1042.618016,1020.019125,1029.296690
1,ElasticNet optuna GroupKFold,1042.622839,1019.958835,1029.331516
2,ElasticNet optuna TimeSeriesSplit,1042.641138,1019.990267,1029.331801


,model,train,valid,test
0,ElasticNet optuna MinMaxScaler,0.581586,0.565138,0.588316
1,ElasticNet optuna GroupKFold,0.581582,0.565189,0.588288
2,ElasticNet optuna TimeSeriesSplit,0.581568,0.565162,0.588288
